<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">05 Mobile AI</h1>
<hr>

## **19. Grand Finale: Know When You Don't Know**

<div class="alert alert-light">

  **Team name:** (add your team name here)

  **Members and matriculation numbers:** (add them here)

</div>

### 19.1 Introduction

This is the last notebook of the chapter, and it is a competition. In teams of four or five, you build a quiz player on your Raspberry Pi: a Python function that answers questions with the models on the Pi. The scoring makes it harder than it sounds. A wrong answer costs more than a right answer earns, and some questions are about things that do not exist. A player that answers everything with confidence loses. The winner is the team whose Pi knows best when it knows the answer.

This is the problem behind hallucination ([05.13](05.13_Experimenting_with_Prompts.ipynb)). A model that is rewarded for every answer and never for "I don't know" learns to guess, and one recent analysis argues that this is a main reason why language models make things up [01]. In this quiz, guessing does not pay.

Everything you need comes from earlier notebooks: probabilities ([05.05](05.05_Introduction_to_LLMs.ipynb)), settings and seeds ([05.12](05.12_Understanding_Inference_Control.ipynb)), prompts ([05.13](05.13_Experimenting_with_Prompts.ipynb)), decision models ([05.14](05.14_New_Kinds_of_Language_Models.ipynb)) and several models working together ([05.17](05.17_Local_Agents_II.ipynb)). How you combine them is up to you.

At the finale, every team presents its player. The three best scores on a hidden test take the podium, and there is a prize for the best idea.

By the end, you should be able to measure how often a model is right, find a signal that tells right answers from wrong ones, decide when an answer is worth giving, and defend a design with your own numbers.

#### 19.1.1 Environment Setup

Everything runs on your Pi through `caspi`, connected as in [05.04_GettingStarted_with_Raspberry_Pi.ipynb](05.04_GettingStarted_with_Raspberry_Pi.ipynb). Your team has several Pis: use them to test different ideas at the same time.

<div class="alert alert-block alert-success">

If a library is missing, use the installation instructions in [00.1_Check_Course_Environment.ipynb](../00_Intro/00.1_Check_Course_Environment.ipynb). Select that Python environment as this notebook's kernel.
</div>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import caspi

caspi.connect()

### 19.2 The Rules

Your player is one function, `answer(question)`. It gets a question as a string and returns one of three things:

- a short answer of at most 8 words, such as `"Bern"` or `"1969"`
- `"does not exist"`, if the question is about something that does not exist
- `None`, to pass

The hidden test has 50 questions, asked in the same order for every team, and **about four minutes in total**: 5 seconds per question on average. Questions that are not answered when the time is up count as passed. The first call is a warm-up that loads your models, and it is not timed.

The exact points stay secret until the finale, but this much is fixed: a right answer earns points, a wrong answer costs more than a right one earns, and a pass costs nothing. An answer longer than 8 words counts as wrong, and so does a list of guesses such as "Bern or Zurich". If your code fails on a question, that question counts as passed.

Your player may use the models on your Pi through `caspi`, as often as the time allows, together with plain Python and `numpy`. It may not use the internet or any other computer, and it may not contain answers written into the code: it has to work on questions it has never seen.

Three limits of the Pi matter here. It keeps at most two models in memory, so a player that switches between three models loads one of them again every time, which takes minutes. Two large models together can be too much for its 8 GB: the Pi then freezes and restarts, as the warning in [05.04](05.04_GettingStarted_with_Raspberry_Pi.ipynb), Section 4.5.2, describes. Run `caspi.stop()` before you try a large model, and build a player that never needs two large ones at once. And it answers one request at a time, so asking several questions in parallel does not make anything faster.

### 19.3 The Practice Test

`caspi.QUIZ` holds 40 practice questions with their accepted answers, mixed like the hidden test. A few of them:

In [ ]:
for question, accepted in caspi.QUIZ[:6]:
    print(f"{question:60s} {accepted or 'does not exist'}")

`caspi.quiz(answer)` plays the whole practice test with your function, under the rules of the hidden test: a warm-up, then 5 seconds per question on average. It prints one line per question and a summary, and it returns the results, so you can analyse them. A reply that matches none of the accepted spellings is marked `wrong?`; check it by eye. The quiz counts right and wrong answers, but it does not give points.

The simplest player passes on everything:

In [ ]:
results = caspi.quiz(lambda question: None)

It is fast, it is never wrong, and it scores nothing. Every player you build has to beat this one.

### 19.4 A Baseline That Works

The next player asks `granite4.2:3b` every question and returns whatever it says. The system prompt keeps the answers short. The practice test takes up to three and a half minutes.

In [ ]:
MODEL = "granite4.2:3b"
caspi.load(MODEL)


def baseline(question):
    reply = caspi.ask(MODEL, question, system="Answer with as few words as possible.",
                      think=False, temperature=0, num_predict=20, show=False)
    return reply.text


results = caspi.quiz(baseline)

Look at the questions about things that do not exist. Does the baseline notice, or does it come up with a name, a year or a number? Then count its right and wrong answers. Say a right answer earns one point: how much may a wrong answer cost before the baseline scores below zero, the score of the player that passes on everything?

`results` holds one row per question, and it is where your evidence comes from:

In [ ]:
table = pd.DataFrame(results)
table[table["verdict"] != "right"]

### 19.5 Ideas

There is no single right design, and none of these ideas is enough on its own. Each one costs time from your budget, so measure before you decide.

#### 19.5.1 Let the model say it does not know

A system prompt can allow the model to pass, or to say that something does not exist ([05.13](05.13_Experimenting_with_Prompts.ipynb), Section 13.6). It is the cheapest idea, and it is rarely enough: a model that makes up a mayor can also make up that it is sure.

#### 19.5.2 Ask several times

With different seeds and a temperature above 0, a model that knows the answer tends to give the same one every time, while a model that guesses gives different ones ([05.12](05.12_Understanding_Inference_Control.ipynb), Section 12.3) [04]. Agreement is a signal, and it costs one call per sample.

```python
replies = [caspi.ask(MODEL, question, seed=seed, temperature=0.8, think=False, show=False).text
           for seed in range(3)]
```

#### 19.5.3 Read the model's confidence

The probability of the first token tells you how sure the model was when it started its answer ([05.05](05.05_Introduction_to_LLMs.ipynb), [05.13](05.13_Experimenting_with_Prompts.ipynb) Section 13.6) [02]. Keep the answers short, so that the first token is part of the answer and not a polite "The".

```python
reply = caspi.ask(MODEL, question, logprobs=True, think=False, show=False)
first = np.exp(reply.logprobs[0]["logprob"])
```

#### 19.5.4 Ask a judge

A decision model returns a probability instead of text ([05.14](05.14_New_Kinds_of_Language_Models.ipynb)). It can judge whether an answer fits the question [05].

```python
text = reply.text   # the answer you want to check
check = {"type": "choice", "instructions": "Is this answer to the question correct?",
         "criteria": {"correct": "the answer is true", "wrong": "the answer is false or made up"}}
decision = caspi.ollama("/v1/systemone", {"model": "tev1:0.8b", "state": f"Question: {question}\nAnswer: {text}",
                                         "questions": {"check": check}})
p_correct = decision["answers"]["check"]["probabilities"]["correct"]
```

#### 19.5.5 Let a second model check

One model answers and a second one, from another family, reviews the answer, like the writer and the reviewer in [05.17](05.17_Local_Agents_II.ipynb). Two small models fit into memory side by side. Two large ones may not, see Section 19.2.

#### 19.5.6 Start small, ask big only when needed

`qwen3.5:0.8b` is fast, and `qwen3.5:4b` knows more but takes longer. Ask the small model first and the large one only when the small one is unsure [03]. A phone makes the same decision when it chooses between answering on the device and sending a request to the cloud.

Some questions name a village, a person or a product that does not exist. Which of these signals notice that?

### 19.6 Build and Measure

A plan that works for most teams:

1. Everyone runs the baseline on their own Pi and reads the wrong answers.
2. Each of you tries one idea from Section 19.5 on your own Pi.
3. You compare the counts, keep what works and combine it.
4. You choose every threshold from a measurement, not from a feeling.

#### 19.6.1 Measure a signal

A signal is useful when it tells right answers from wrong ones. The next cell shows how to check that, with the first-token probability of the baseline as the example. The player stores the signal for every question it answers, and the plot shows where the right and the wrong answers fall.

In [ ]:
first_token = {}   # question -> probability of the first token


def baseline_with_confidence(question):
    reply = caspi.ask(MODEL, question, system="Answer with as few words as possible.",
                      think=False, temperature=0, num_predict=20, logprobs=True, show=False)
    first_token[question] = float(np.exp(reply.logprobs[0]["logprob"]))
    return reply.text


def show_signal(results, signal, label):
    """Where the right and the wrong answers fall on one signal."""
    right = [signal[r["question"]] for r in results if r["verdict"] == "right"]
    wrong = [signal[r["question"]] for r in results if r["verdict"] in ("wrong", "wrong?")]
    plt.hist([right, wrong], bins=10, range=(0, 1), color=["#1b9e77", "#d95f02"], label=["right", "wrong"])
    plt.xlabel(label)
    plt.ylabel("answers")
    plt.legend()
    plt.show()


results = caspi.quiz(baseline_with_confidence)
show_signal(results, first_token, "probability of the first token")

If the wrong answers sit on the left and the right ones on the right, a threshold between them separates many of them. If the two overlap everywhere, this signal does not help this model. The same plot works for any signal you invent: agreement between samples, the judge's probability, or a mix.

How sure must a player be before an answer pays off? That depends on how many points a right answer earns and a wrong one costs. Question 03 asks you to work it out.

#### 19.6.2 Do not fit the practice test

The practice test is small, and the hidden questions are different. A rule that you tune until it gets every practice question right has learned these 40 questions, not when to trust a model. Tune on one half and check on the other:

In [ ]:
tune, check = caspi.QUIZ[::2], caspi.QUIZ[1::2]


def answer(question):
    return None   # your idea here


results = caspi.quiz(answer, tune)

When you are happy with the result on `tune`, run `caspi.quiz(answer, check)` once. If your player does much better on `tune` than on `check`, it is fitted to the questions, not to the problem.

### 19.7 Your Player

When you have decided, put your player into the next cell. This cell is what you submit, so it has to stand on its own:

- it starts with the line `# OUR PLAYER`
- it imports everything it uses
- it defines `answer(question)`; helper functions and constants in the same cell are fine
- it uses nothing from other cells and reads no files

We run it on a course Pi with the hidden questions, exactly the way `caspi.quiz` runs the practice test. It starts as a copy of the baseline: replace it with yours.

In [ ]:
# OUR PLAYER
import caspi

MODEL = "granite4.2:3b"


def answer(question):
    reply = caspi.ask(MODEL, question, system="Answer with as few words as possible.",
                      think=False, temperature=0, num_predict=20, show=False)
    return reply.text

Run the whole practice test one last time with this cell. Fix every error now: in the hidden run, an error counts as a pass.

In [ ]:
results = caspi.quiz(answer)

### 19.8 The Finale

At the finale, every team presents its player in a few minutes: the idea, the numbers that convinced you, and one thing that did not work. Then we reveal the hidden questions, the points and the podium: the three best scores, and a prize for the best idea.

### 19.9 Further Reading and Literature

[01] Kalai, A. T., Nachum, O., Vempala, S. S., & Zhang, E. (2025). **Why Language Models Hallucinate.** [Paper](https://arxiv.org/abs/2509.04664). Argues that tests which reward guessing over "I don't know" teach models to guess.

[02] Kadavath, S., Conerly, T., Askell, A., et al. (2022). **Language Models (Mostly) Know What They Know.** [Paper](https://arxiv.org/abs/2207.05221). Whether a model's own probabilities show when it is wrong.

[03] Chen, L., Zaharia, M., & Zou, J. (2023). **FrugalGPT: How to Use Large Language Models While Reducing Cost and Improving Performance.** [Paper](https://arxiv.org/abs/2305.05176). Asking a cheap model first and a larger one only when needed.

[04] Wang, X., Wei, J., Schuurmans, D., et al. (2023). **Self-Consistency Improves Chain of Thought Reasoning in Language Models.** ICLR. [Paper](https://arxiv.org/abs/2203.11171)

[05] Zheng, L., Chiang, W.-L., Sheng, Y., et al. (2023). **Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena.** NeurIPS Datasets and Benchmarks. [Paper](https://arxiv.org/abs/2306.05685)

---
### 19.10 Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

Use your own outputs as evidence. Copy the relevant numbers into your answer so that they remain visible after saving.

**Question 01: Your player**

Describe your player in a few sentences: which models, which signals, which thresholds. Why this design and not another?

(answer here)

**Question 02: Baseline and player**

Report the practice-test counts of the baseline and of your final player: right, wrong, passed, not reached, and the time. What changed most?

(answer here)

**Question 03: When to answer**

Suppose a right answer earns r points and a wrong one costs w points. From which probability of being right does answering pay off? How did your player decide when to answer?

(answer here)

**Question 04: Your signal**

Which signal told right answers from wrong ones best, and how did you measure it? Describe your plot from Section 19.6.1, or the one you made for your own signal.

(answer here)

**Question 05: Things that do not exist**

How did your player handle the questions about things that do not exist? Which one fooled it, and why might that be?

(answer here)

**Question 06: The time budget**

Where did your player spend its time? What did you give up to stay within 5 seconds per question?

(answer here)

**Question 07: Tuned or fitted**

Compare your player on `tune` and on `check` from Section 19.6.2. Is it fitted to the practice questions? What do you expect on the hidden test?

(answer here)

**Question 08: An idea that did not work**

Describe one idea you tried and dropped, with the numbers that made you drop it.

(answer here)

**Question 09: Beyond the quiz**

Think of a system in your organisation that answers questions automatically. Where would a wrong answer cost more than no answer, and how would you decide when it should answer?

(answer here)

**Question 10: Optional: after the finale**

How did your score on the hidden test compare with the practice test? Explain the difference.

(answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)